# Resolution Detector — True Win Rate

**Strategy**: Single DuckDB scan → filtered table (~1% of data) → all SQL analysis.

**Runtime**: ~10-15 min for the parquet scan, then seconds for everything else.

In [1]:
import pathlib, duckdb, time, requests, pandas as pd

for _b in [pathlib.Path.cwd(), pathlib.Path.cwd().parent, pathlib.Path.cwd().parent.parent]:
    if (_b / 'data' / 'parquet').exists():
        PQ = str(_b / 'data' / 'parquet' / 'polymarket_*.parquet')
        CSV = str(_b / 'data' / 'market_metadata.csv')
        DB = str(_b / 'data' / 'updown.duckdb')
        break

con = duckdb.connect(DB)
con.execute("SET memory_limit='4GB'")
con.execute("SET threads=4")
con.execute("SET preserve_insertion_order=false")
print('OK')

OK


## 1. One-Time Scan: Build Filtered Table

Scans all 617 parquet files but only keeps rows matching "up or down" asset IDs.
Result is persisted in `data/updown.duckdb` — subsequent runs skip this cell.

In [2]:
%%time
exists = con.execute(
    "SELECT COUNT(*) FROM information_schema.tables WHERE table_name='ticks'"
).fetchone()[0]

if exists:
    n = con.execute("SELECT COUNT(*) FROM ticks").fetchone()[0]
    print(f'Table exists with {n:,} rows — skipping scan.')
else:
    print('Building filtered table (one-time scan)...')
    con.execute(f"""
        CREATE TABLE ticks AS
        SELECT
            epoch_ms(t.timestamp) AS ts_ms,
            t.price, t.best_bid, t.best_ask,
            t.event_type, t.asset_id,
            m.condition_id, m.outcome, m.question, m.end_date
        FROM read_parquet('{PQ}') t
        INNER JOIN (
            SELECT DISTINCT asset_id, condition_id, outcome, question, end_date
            FROM read_csv('{CSV}', types={{'asset_id':'VARCHAR','outcome':'VARCHAR'}})
            WHERE question ILIKE '%up or down%'
              AND COALESCE(taker_fee_bps, 0) > 0
        ) m ON t.asset_id = m.asset_id
    """)
    n = con.execute("SELECT COUNT(*) FROM ticks").fetchone()[0]
    print(f'Created table: {n:,} rows')
    # Skip indexes — they OOM on 520M rows with 4GB limit
    # DuckDB's columnar scans are fast enough without them
    print('Done (no indexes — columnar scan is fast enough).')

Table exists with 519,997,319 rows — skipping scan.
CPU times: user 11 ms, sys: 2.14 ms, total: 13.1 ms
Wall time: 14.4 ms


## 2. Detect Resolutions + Entry Signals (pure SQL)

In [3]:
%%time
THRESHOLD = 0.40

# Everything in one SQL: resolutions, entries, merged
merged = con.execute(f"""
    -- 1. Last NO book per condition (resolution detection)
    WITH last_no_book AS (
        SELECT DISTINCT ON (condition_id)
            condition_id, ts_ms, best_bid, best_ask,
            (best_bid + best_ask) / 2.0 AS last_mid,
            question, end_date
        FROM ticks
        WHERE outcome = 'NO' AND event_type = 'PRICE_CHANGE'
          AND best_bid > 0 AND best_ask > best_bid
        ORDER BY condition_id, ts_ms DESC
    ),
    -- 2. Last NO trade per condition
    last_no_trade AS (
        SELECT DISTINCT ON (condition_id)
            condition_id, price AS last_trade_px
        FROM ticks
        WHERE outcome = 'NO' AND event_type = 'LAST_TRADE'
        ORDER BY condition_id, ts_ms DESC
    ),
    -- 3. Resolution classification
    resolutions AS (
        SELECT
            b.condition_id, b.last_mid, b.question, b.end_date,
            COALESCE(t.last_trade_px, 0) AS last_trade_px,
            CASE
                WHEN b.last_mid >= 0.95 OR COALESCE(t.last_trade_px, 0) >= 0.95 THEN 'NO_WINS'
                WHEN b.last_mid <= 0.05 OR COALESCE(t.last_trade_px, 0) <= 0.05 THEN 'YES_WINS'
                ELSE 'UNRESOLVED'
            END AS resolution
        FROM last_no_book b
        LEFT JOIN last_no_trade t USING (condition_id)
    ),
    -- 4. First YES mid cross below threshold per condition
    first_cross AS (
        SELECT DISTINCT ON (condition_id)
            condition_id, ts_ms AS cross_ts
        FROM ticks
        WHERE outcome = 'YES' AND event_type = 'PRICE_CHANGE'
          AND best_bid > 0 AND best_ask > best_bid
          AND (best_bid + best_ask) / 2.0 <= {THRESHOLD}
        ORDER BY condition_id, ts_ms
    ),
    -- 5. NO asset per condition
    no_map AS (
        SELECT DISTINCT condition_id,
               FIRST(asset_id) AS no_asset
        FROM ticks WHERE outcome = 'NO'
        GROUP BY condition_id
    ),
    -- 6. NO book nearest to cross time (within 5s)
    no_at_cross AS (
        SELECT DISTINCT ON (fc.condition_id)
            fc.condition_id,
            fc.cross_ts,
            nm.no_asset,
            t.best_bid AS entry_bid,
            t.best_ask AS entry_ask,
            (t.best_bid + t.best_ask) / 2.0 AS entry_mid
        FROM first_cross fc
        JOIN no_map nm USING (condition_id)
        JOIN ticks t ON t.asset_id = nm.no_asset
            AND t.event_type = 'PRICE_CHANGE'
            AND t.ts_ms BETWEEN fc.cross_ts - 5000 AND fc.cross_ts + 5000
            AND t.best_bid > 0 AND t.best_ask > t.best_bid
        ORDER BY fc.condition_id, ABS(t.ts_ms - fc.cross_ts)
    ),
    -- 7. Tick span per condition (for duration analysis)
    spans AS (
        SELECT condition_id,
               MIN(ts_ms) AS first_ts, MAX(ts_ms) AS last_ts,
               (MAX(ts_ms) - MIN(ts_ms)) / 60000.0 AS span_min
        FROM ticks GROUP BY condition_id
    )
    -- Final merge: entries + resolutions + durations
    SELECT
        nc.condition_id,
        nc.cross_ts AS entry_ts,
        nc.entry_ask,
        nc.entry_bid,
        nc.entry_mid,
        r.resolution,
        r.last_mid,
        r.last_trade_px,
        r.question,
        r.end_date,
        COALESCE(s.span_min, 0) AS span_min
    FROM no_at_cross nc
    JOIN resolutions r USING (condition_id)
    LEFT JOIN spans s USING (condition_id)
    ORDER BY nc.cross_ts
""").fetchdf()

print(f"Entry signals: {len(merged)}")
print(f"Avg entry ask: ${merged['entry_ask'].mean():.3f}")
print(f"\nTick-data resolution:")
for k, v in merged['resolution'].value_counts().items():
    print(f"  {k:15s}: {v:>5d} ({v/len(merged)*100:5.1f}%)")

res = merged[merged['resolution'] != 'UNRESOLVED']
if len(res) > 0:
    nw = (res['resolution'] == 'NO_WINS').sum()
    yw = (res['resolution'] == 'YES_WINS').sum()
    print(f"\nTick-data win rate: {nw}/{nw+yw} = {nw/(nw+yw)*100:.1f}%")

Entry signals: 2858
Avg entry ask: $0.624

Tick-data resolution:
  NO_WINS        :  1770 ( 61.9%)
  YES_WINS       :  1066 ( 37.3%)
  UNRESOLVED     :    22 (  0.8%)

Tick-data win rate: 1770/2836 = 62.4%
CPU times: user 1min 15s, sys: 16 s, total: 1min 31s
Wall time: 40.5 s


## 3. API Lookup for Unresolved Markets

In [4]:
unresolved_cids = merged[merged['resolution'] == 'UNRESOLVED']['condition_id'].unique()
print(f"Querying API for {len(unresolved_cids)} unresolved conditions...")

api_map = {}
GAMMA = "https://gamma-api.polymarket.com"

for i, cid in enumerate(unresolved_cids):
    try:
        r = requests.get(f"{GAMMA}/markets", params={'condition_id': cid}, timeout=10)
        if r.status_code == 200:
            d = r.json()
            m = d[0] if isinstance(d, list) and d else d if isinstance(d, dict) else None
            if m:
                o = str(m.get('outcome', '')).strip().upper()
                if o in ('NO', 'FALSE', '0'):   api_map[cid] = 'NO_WINS'
                elif o in ('YES', 'TRUE', '1'): api_map[cid] = 'YES_WINS'
                elif m.get('closed') or m.get('resolved'): api_map[cid] = 'RESOLVED_UNKNOWN'
                else: api_map[cid] = 'STILL_OPEN'
            else: api_map[cid] = 'NOT_FOUND'
        else: api_map[cid] = f'HTTP_{r.status_code}'
    except: api_map[cid] = 'ERROR'
    if (i+1) % 5 == 0: time.sleep(1)
    if (i+1) % 100 == 0: print(f"  {i+1}/{len(unresolved_cids)}...")

print(f"\nAPI results:")
from collections import Counter
for k, v in Counter(api_map.values()).most_common():
    print(f"  {k:20s}: {v}")

# Merge
merged['final'] = merged.apply(
    lambda r: r['resolution'] if r['resolution'] != 'UNRESOLVED'
              else api_map.get(r['condition_id'], 'UNKNOWN'), axis=1)

print(f"\n{'='*55}")
print("FINAL RESOLUTION (tick + API)")
print(f"{'='*55}")
for k, v in merged['final'].value_counts().items():
    print(f"  {k:20s}: {v:>5d} ({v/len(merged)*100:5.1f}%)")

rf = merged[merged['final'].isin(['NO_WINS', 'YES_WINS'])]
if len(rf) > 0:
    nw = (rf['final'] == 'NO_WINS').sum()
    yw = (rf['final'] == 'YES_WINS').sum()
    print(f"\n  TRUE WIN RATE: {nw}/{nw+yw} = {nw/(nw+yw)*100:.1f}%")

Querying API for 22 unresolved conditions...

API results:
  RESOLVED_UNKNOWN    : 22

FINAL RESOLUTION (tick + API)
  NO_WINS             :  1770 ( 61.9%)
  YES_WINS            :  1066 ( 37.3%)
  RESOLVED_UNKNOWN    :    22 (  0.8%)

  TRUE WIN RATE: 1770/2836 = 62.4%


## 4. True PnL + Duration + Kelly

In [6]:
import numpy as np

SIZE, SLIP = 10.0, 0.0005
FR, FE = 0.25, 2  # fee_rate, fee_exponent

ep = merged['entry_ask'].values + SLIP
cost = ep * SIZE
fb = FR * (ep * (1 - ep)) ** FE  # fee base at entry
buy_fee = SIZE * fb * ep

# Vectorized PnL
final = merged['final'].values
last_mid = merged['last_mid'].values

pnl = np.where(
    final == 'NO_WINS',
    SIZE * 1.0 - cost - buy_fee - SIZE * FR * (1.0 * 0.0) ** FE,  # fee at p=1 is 0
    np.where(
        final == 'YES_WINS',
        -cost - buy_fee,
        # Open: MTM
        last_mid * SIZE - cost - buy_fee - SIZE * FR * (last_mid * (1 - last_mid)) ** FE
    )
)

status = np.where(final == 'NO_WINS', 'WIN',
         np.where(final == 'YES_WINS', 'LOSS', 'OPEN'))

merged['pnl'] = pnl
merged['status'] = status
merged['entry_price'] = ep
merged['cost'] = cost

# ── PnL table ──
print('=' * 60)
print(f"{'Status':<12} {'Count':>6} {'Total PnL':>12} {'Avg PnL':>10}")
print('-' * 60)
for st in ['WIN', 'LOSS', 'OPEN']:
    m = merged[merged['status'] == st]
    if len(m): print(f"{st:<12} {len(m):>6} {m['pnl'].sum():>12.2f} {m['pnl'].mean():>10.3f}")
print('-' * 60)
print(f"{'TOTAL':<12} {len(merged):>6} {merged['pnl'].sum():>12.2f} {merged['pnl'].mean():>10.3f}")
print('=' * 60)

rp = merged[merged['status'].isin(['WIN', 'LOSS'])]
w, l = rp[rp['status']=='WIN'], rp[rp['status']=='LOSS']
if len(rp):
    print(f"\nResolved PnL: ${rp['pnl'].sum():.2f}  |  Win rate: {len(w)}/{len(rp)} = {len(w)/len(rp)*100:.1f}%")
    if len(w): print(f"Avg win:  ${w['pnl'].mean():.3f}")
    if len(l): print(f"Avg loss: ${l['pnl'].mean():.3f}")

# ── Duration breakdown ──
def bkt(m):
    if m <= 0: return 'unknown'
    if m <= 10: return '0-10min'
    if m <= 30: return '10-30min'
    if m <= 60: return '30-60min'
    if m <= 240: return '1-4h'
    if m <= 1440: return '4-24h'
    return '24h+'

merged['bucket'] = merged['span_min'].apply(bkt)

print(f"\n{'='*65}")
print('PnL BY MARKET DURATION')
print(f"{'='*65}")
print(f"{'Dur':<12} {'N':>5} {'Win':>4} {'Loss':>5} {'WR':>6} {'PnL':>10} {'Avg':>8}")
print('-' * 65)
for b in ['0-10min','10-30min','30-60min','1-4h','4-24h','24h+','unknown']:
    s = merged[merged['bucket'] == b]
    if not len(s): continue
    sw = (s['status']=='WIN').sum(); sl = (s['status']=='LOSS').sum()
    r = sw + sl
    wr = f"{sw/r*100:.0f}%" if r else 'N/A'
    print(f"{b:<12} {len(s):>5} {sw:>4} {sl:>5} {wr:>6} {s['pnl'].sum():>10.2f} {s['pnl'].mean():>8.3f}")

print('\nCapital efficiency:')
for b in ['0-10min','10-30min','30-60min','1-4h','4-24h','24h+']:
    s = merged[merged['bucket'] == b]
    if not len(s): continue
    ch = (s['cost'] * s['span_min'] / 60).sum()
    if ch > 0:
        print(f"  {b:<12}: avg {s['span_min'].mean()/60:.1f}h, PnL/cap-hr = ${s['pnl'].sum()/ch:.4f}")

# ── Kelly ──
if len(rp) and len(w) and len(l):
    p = len(w) / len(rp)
    aw, al = w['pnl'].mean(), abs(l['pnl'].mean())
    br = aw / al if al > 0 else 0
    k = p - (1-p) / br if br > 0 else -1
    print(f"\n{'='*55}")
    print('KELLY CRITERION')
    print(f"{'='*55}")
    print(f"  WIN: n={len(w)}, avg entry=${w['entry_price'].mean():.3f}, range=[${w['entry_price'].min():.3f}, ${w['entry_price'].max():.3f}]")
    print(f"  LOSS: n={len(l)}, avg entry=${l['entry_price'].mean():.3f}, range=[${l['entry_price'].min():.3f}, ${l['entry_price'].max():.3f}]")
    print(f"  Win rate: {p*100:.1f}%  |  Avg win: ${aw:.3f}  |  Avg loss: ${al:.3f}")
    print(f"  W/L ratio: {br:.3f}  |  Kelly f*: {k:.4f}  |  Half: {k/2:.4f}")
    if k <= 0:
        print(f"  >>> NEGATIVE KELLY — NO EDGE <<<")
    else:
        print(f"  >>> Optimal: {k*100:.1f}% of capital ({k/2*100:.1f}% half-Kelly)")
        print(f"  >>> On $1000: ${k*1000:.0f}/trade (half: ${k/2*1000:.0f})")

Status        Count    Total PnL    Avg PnL
------------------------------------------------------------
WIN            1770      6486.15      3.664
LOSS           1066     -6721.31     -6.305
OPEN             22       -25.03     -1.138
------------------------------------------------------------
TOTAL          2858      -260.19     -0.091

Resolved PnL: $-235.17  |  Win rate: 1770/2836 = 62.4%
Avg win:  $3.664
Avg loss: $-6.305

PnL BY MARKET DURATION
Dur              N  Win  Loss     WR        PnL      Avg
-----------------------------------------------------------------
0-10min         76   48    28    63%     -25.52   -0.336
10-30min       144   92    51    64%      16.67    0.116
30-60min       160  111    49    69%      99.45    0.622
1-4h           297  166   127    57%    -193.40   -0.651
4-24h         2056 1278   762    63%    -128.40   -0.062
24h+           125   75    49    60%     -28.99   -0.232

Capital efficiency:
  0-10min     : avg 0.1h, PnL/cap-hr = $-0.4583
  10-30mi

## 5. Summary

## 6. Resolution Sniping Analysis

In the final seconds before a market resolves, the outcome is often certain from the underlying price — but tokens still trade at $0.90-$0.95 instead of $1.00. Measure that gap.

In [8]:
%%time
# For every market that resolved NO_WINS, look at the NO token price
# in the final 5min, 2min, 1min, 30s of activity.
# The gap between that price and $1.00 is the sniping opportunity.

snipe = con.execute("""
    -- 1. Markets that resolved to NO_WINS (from our earlier analysis)
    WITH resolved_no AS (
        SELECT DISTINCT ON (condition_id)
            condition_id,
            ts_ms AS resolve_ts
        FROM ticks
        WHERE outcome = 'NO' AND event_type = 'PRICE_CHANGE'
          AND best_bid > 0 AND best_ask > best_bid
          AND (best_bid + best_ask) / 2.0 >= 0.95
        ORDER BY condition_id, ts_ms
        -- first time NO hits 0.95+ = resolution moment
    ),
    -- 2. Also get the LAST tick per condition (data boundary)
    last_tick AS (
        SELECT condition_id, MAX(ts_ms) AS last_ts
        FROM ticks
        GROUP BY condition_id
    ),
    -- 3. NO token book updates in the minutes BEFORE resolution
    pre_resolve AS (
        SELECT
            t.condition_id,
            t.ts_ms,
            t.best_bid, t.best_ask,
            (t.best_bid + t.best_ask) / 2.0 AS mid,
            r.resolve_ts,
            lt.last_ts,
            (r.resolve_ts - t.ts_ms) / 1000.0 AS secs_before_resolve
        FROM ticks t
        JOIN resolved_no r USING (condition_id)
        JOIN last_tick lt USING (condition_id)
        WHERE t.outcome = 'NO'
          AND t.event_type = 'PRICE_CHANGE'
          AND t.best_bid > 0 AND t.best_ask > t.best_bid
          AND t.ts_ms < r.resolve_ts
          AND t.ts_ms >= r.resolve_ts - 300000  -- last 5 minutes before resolution
    ),
    -- 4. Bucket into time windows and get the LAST quote in each window
    bucketed AS (
        SELECT *,
            CASE
                WHEN secs_before_resolve <= 30 THEN '0-30s'
                WHEN secs_before_resolve <= 60 THEN '30-60s'
                WHEN secs_before_resolve <= 120 THEN '1-2min'
                WHEN secs_before_resolve <= 300 THEN '2-5min'
            END AS time_bucket
        FROM pre_resolve
    )
    SELECT
        time_bucket,
        COUNT(DISTINCT condition_id) AS n_markets,
        AVG(mid) AS avg_no_mid,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY mid) AS p25_mid,
        PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY mid) AS p50_mid,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY mid) AS p75_mid,
        AVG(best_ask) AS avg_ask,
        AVG(1.0 - best_ask) AS avg_discount,
        -- How many shares could you buy? (proxy: count of quotes with ask < 0.98)
        SUM(CASE WHEN best_ask < 0.98 THEN 1 ELSE 0 END) AS cheap_quotes,
        COUNT(*) AS total_quotes
    FROM bucketed
    WHERE time_bucket IS NOT NULL
    GROUP BY time_bucket
    ORDER BY time_bucket
""").fetchdf()

print("=== PRE-RESOLUTION NO TOKEN PRICING ===")
print("(markets that resolved NO_WINS — what was the NO price BEFORE resolution?)\n")

for _, row in snipe.iterrows():
    tb = row['time_bucket']
    n = int(row['n_markets'])
    avg_ask = row['avg_ask']
    disc = row['avg_discount']
    cheap = int(row['cheap_quotes'])
    total = int(row['total_quotes'])
    print(f"  {tb:>8s}:  {n:>4d} markets | avg NO ask=${avg_ask:.3f} | "
          f"discount=${disc:.3f}/sh | "
          f"mid p25/p50/p75: ${row['p25_mid']:.3f}/${row['p50_mid']:.3f}/${row['p75_mid']:.3f} | "
          f"cheap quotes: {cheap}/{total}")

print(f"\n  'discount' = $1.00 - ask = how much you'd profit per share by buying at ask")
print(f"  'cheap quotes' = quotes where ask < $0.98 (>2¢ profit opportunity)")

=== PRE-RESOLUTION NO TOKEN PRICING ===
(markets that resolved NO_WINS — what was the NO price BEFORE resolution?)

     0-30s:  1841 markets | avg NO ask=$0.769 | discount=$0.231/sh | mid p25/p50/p75: $0.685/$0.835/$0.905 | cheap quotes: 10302127/10304145
    1-2min:  1834 markets | avg NO ask=$0.609 | discount=$0.391/sh | mid p25/p50/p75: $0.470/$0.635/$0.760 | cheap quotes: 22593811/22593824
    2-5min:  1831 markets | avg NO ask=$0.547 | discount=$0.453/sh | mid p25/p50/p75: $0.425/$0.535/$0.660 | cheap quotes: 47568441/47568482
    30-60s:  1837 markets | avg NO ask=$0.672 | discount=$0.328/sh | mid p25/p50/p75: $0.545/$0.715/$0.825 | cheap quotes: 11434937/11434981

  'discount' = $1.00 - ask = how much you'd profit per share by buying at ask
  'cheap quotes' = quotes where ask < $0.98 (>2¢ profit opportunity)
CPU times: user 58.6 s, sys: 14.2 s, total: 1min 12s
Wall time: 42.7 s


In [2]:
%%time
# BOTH SIDES: snipe YES tokens before YES_WINS resolution too
# + compute hourly rate to see if this is scalable

both_sides = con.execute("""
    -- NO side: buy NO before NO resolves
    WITH no_resolve AS (
        SELECT DISTINCT ON (condition_id)
            condition_id, ts_ms AS resolve_ts, 'NO' AS winning_side
        FROM ticks
        WHERE outcome = 'NO' AND event_type = 'PRICE_CHANGE'
          AND best_bid > 0 AND best_ask > best_bid
          AND (best_bid + best_ask) / 2.0 >= 0.95
        ORDER BY condition_id, ts_ms
    ),
    -- YES side: buy YES before YES resolves
    yes_resolve AS (
        SELECT DISTINCT ON (condition_id)
            condition_id, ts_ms AS resolve_ts, 'YES' AS winning_side
        FROM ticks
        WHERE outcome = 'YES' AND event_type = 'PRICE_CHANGE'
          AND best_bid > 0 AND best_ask > best_bid
          AND (best_bid + best_ask) / 2.0 >= 0.95
        ORDER BY condition_id, ts_ms
    ),
    all_resolves AS (
        SELECT * FROM no_resolve
        UNION ALL
        SELECT * FROM yes_resolve
    ),
    -- Last quote for the WINNING token before resolution
    snipe_quotes AS (
        SELECT DISTINCT ON (r.condition_id, r.winning_side)
            r.condition_id,
            r.winning_side,
            r.resolve_ts,
            t.ts_ms AS quote_ts,
            t.best_ask,
            t.best_bid,
            (t.best_bid + t.best_ask) / 2.0 AS mid,
            (r.resolve_ts - t.ts_ms) / 1000.0 AS secs_before
        FROM all_resolves r
        JOIN ticks t ON t.condition_id = r.condition_id
            AND t.outcome = r.winning_side
            AND t.event_type = 'PRICE_CHANGE'
            AND t.best_bid > 0 AND t.best_ask > t.best_bid
            AND t.ts_ms < r.resolve_ts
            AND t.ts_ms >= r.resolve_ts - 60000
        ORDER BY r.condition_id, r.winning_side, t.ts_ms DESC
    )
    SELECT * FROM snipe_quotes
    WHERE best_ask < 0.95
    ORDER BY resolve_ts
""").fetchdf()

if len(both_sides) > 0:
    SIZE = 10.0
    ep = both_sides['best_ask'].values + 0.0005
    fb = 0.25 * (ep * (1 - ep)) ** 2
    profit = SIZE * (1.0 - ep) - SIZE * fb * ep  # profit = (1 - ask) * shares - fee

    both_sides['profit'] = profit

    # Time span of data
    min_ts = both_sides['resolve_ts'].min()
    max_ts = both_sides['resolve_ts'].max()
    hours = (max_ts - min_ts) / 3.6e6

    print(f"=== BOTH-SIDES RESOLUTION SNIPING ===")
    print(f"Snipe opportunities (ask < $0.95, last 60s): {len(both_sides)}")
    print(f"  NO-side snipes:  {(both_sides['winning_side']=='NO').sum()}")
    print(f"  YES-side snipes: {(both_sides['winning_side']=='YES').sum()}")
    print(f"\nTotal profit:      ${profit.sum():.2f}")
    print(f"Avg profit/snipe:  ${profit.mean():.3f}")
    print(f"Data span:         {hours:.1f} hours")
    print(f"Snipes/hour:       {len(both_sides)/max(hours,1):.1f}")
    print(f"Profit/hour:       ${profit.sum()/max(hours,1):.2f}")
    print(f"\nAvg ask:           ${both_sides['best_ask'].mean():.3f}")
    print(f"Avg secs before:   {both_sides['secs_before'].mean():.1f}s")
    print(f"Med secs before:   {both_sides['secs_before'].median():.1f}s")

    # By winning side
    print(f"\n--- By side ---")
    for side in ['NO', 'YES']:
        mask = both_sides['winning_side'] == side
        sub = both_sides[mask]
        p = profit[mask.values]
        if len(sub):
            print(f"  {side}: {len(sub)} snipes, avg ask ${sub['best_ask'].mean():.3f}, "
                  f"profit ${p.sum():.2f} (${p.mean():.3f}/trade)")

    # What if we require ask < 0.90 (bigger discount, safer)?
    print(f"\n--- Sensitivity: different ask thresholds ---")
    for thresh in [0.95, 0.90, 0.85, 0.80, 0.70]:
        mask = both_sides['best_ask'] < thresh
        sub = both_sides[mask]
        p = profit[mask.values]
        if len(sub):
            print(f"  ask < ${thresh:.2f}: {len(sub):>4d} snipes, "
                  f"total ${p.sum():>8.2f}, "
                  f"avg ${p.mean():.3f}/trade, "
                  f"${p.sum()/max(hours,1):.2f}/hr")
else:
    print("No snipe opportunities on either side.")

CPU times: user 15 μs, sys: 13 μs, total: 28 μs
Wall time: 12.9 μs


NameError: name 'con' is not defined

In [ ]:
%%time
# FALSE POSITIVE CHECK: split into two lean queries (NO then YES)
con.execute("SET threads=2")  # reduce memory pressure

fp_results = []
for side in ['NO', 'YES']:
    fp = con.execute(f"""
        WITH peak AS (
            SELECT condition_id,
                   MAX((best_bid + best_ask) / 2.0) AS peak_mid
            FROM ticks
            WHERE outcome = '{side}' AND event_type = 'PRICE_CHANGE'
              AND best_bid > 0 AND best_ask > best_bid
            GROUP BY condition_id
            HAVING peak_mid >= 0.85
        ),
        final AS (
            SELECT DISTINCT ON (condition_id)
                condition_id,
                (best_bid + best_ask) / 2.0 AS final_mid
            FROM ticks
            WHERE outcome = '{side}' AND event_type = 'PRICE_CHANGE'
              AND best_bid > 0 AND best_ask > best_bid
            ORDER BY condition_id, ts_ms DESC
        )
        SELECT
            '{side}' AS side,
            COUNT(*) AS n_peaked,
            SUM(CASE WHEN f.final_mid >= 0.95 THEN 1 ELSE 0 END) AS resolved_win,
            SUM(CASE WHEN f.final_mid < 0.95 THEN 1 ELSE 0 END) AS didnt_resolve,
            SUM(CASE WHEN f.final_mid < 0.50 THEN 1 ELSE 0 END) AS reversed_hard,
            AVG(CASE WHEN f.final_mid < 0.95 THEN f.final_mid END) AS avg_final_if_failed
        FROM peak p
        JOIN final f USING (condition_id)
    """).fetchdf()
    fp_results.append(fp)

con.execute("SET threads=4")
false_positives = pd.concat(fp_results, ignore_index=True)

print("=== FALSE POSITIVE CHECK ===")
print("Tokens that reached mid >= $0.85 at some point:\n")
for _, row in false_positives.iterrows():
    side = row['side']
    n = int(row['n_peaked'])
    won = int(row['resolved_win'])
    failed = int(row['didnt_resolve'])
    rev = int(row['reversed_hard'])
    avg_f = row['avg_final_if_failed']
    print(f"  {side} tokens:")
    print(f"    Peaked >= $0.85:      {n}")
    print(f"    Actually resolved:    {won} ({won/n*100:.1f}%)")
    print(f"    Didn't resolve:       {failed} ({failed/n*100:.1f}%)")
    print(f"    Reversed below $0.50: {rev}")
    if not pd.isna(avg_f):
        print(f"    Avg final if failed:  ${avg_f:.3f}")
    print()

print("If 'Actually resolved' > 95%, sniping at mid >= 0.85 is safe.")
print("If 'Reversed below $0.50' is significant, we'd get destroyed.")